In [ ]:
import torch

from dataset_loaders import build_data_loaders
from utils.checkpoints import load_ae_from_path, load_from_wandb
from utils.config import DatasetConfig
from utils.visualisation import plot_latent_space, show, show_comparison

In [ ]:
ae_path = load_from_wandb("autoencoder_celeba")
ae = load_ae_from_path(ae_path, device=torch.device("mps"))

In [ ]:
dataset_cfg = DatasetConfig(
    name="celeba_single_attribute",
    channels=3,
    height=64,
    width=64,
    num_classes=2,
)

dataloader, test_loader = build_data_loaders(dataset_cfg, batch_size=64, shuffle_test=True, num_workers=0)


In [ ]:
images, labels = next(iter(test_loader))

with torch.no_grad():
    outputs = ae(images)
    recon = torch.sigmoid(outputs.reconstructed)

show_comparison(images, recon)


In [ ]:
from models.autoencoder import AbstractAutoencoder


def ae_latent_space(model: AbstractAutoencoder, data_loader: torch.utils.data.DataLoader,
                    title="Latent Space of CelebA Autoencoder"):
    samples = []
    sampled_labels = []
    i = 0
    for image, label in data_loader:
        latent = model.encode(image)
        samples.append(latent)
        sampled_labels.append(label)
        i = i + 1
        if i > 10:
            break

    samples = torch.cat(samples, dim=0)
    sampled_labels = torch.cat(sampled_labels, dim=0)
    plot_latent_space(samples, sampled_labels, title=title)


In [ ]:
ae_latent_space(ae, test_loader)

In [ ]:
random_latent = torch.rand(16, 128)
with torch.no_grad():
    logits = ae.decode(random_latent)
    images = torch.sigmoid(logits)

show(images, title="Samples from random latents")